## Build a chatbot using LangGraph that remembers the conversation within a session.

### What it should do:

* User types a message
* Agent responds using an LLM
* User types another message — agent remembers the previous one
* Prove memory works by asking "what did I say earlier?"

### Constraints:

* Use MemorySaver as checkpointer
* Use MessagesState for state
* Must use a thread ID in the config when invoking
* No LangChain chains — raw LangGraph only

In [1]:
# python
 
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

True

In [2]:
print("GROQ_API_KEY available:", bool(os.getenv("GROQ_API_KEY")))

GROQ_API_KEY available: True


In [3]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
import uuid # for thread_id

In [4]:
model = init_chat_model(
    model="openai/gpt-oss-120b",       # The specific Groq model ID
    model_provider="groq",        # Specifies the provider
    temperature=0                 # Optional parameters
)

### **Challenge 3** — Add a tool to your agent

* Based on the tool calling session your mentor taught:

What to build: Add a web search tool to your existing chatbot. When the user asks something factual the LLM can't answer confidently, the agent should call the tool and use the result.

### What it should do:

- User asks "what is today's weather in Mumbai?"
- Agent calls the search tool
- Agent returns an answer based on the search result

Reference docs:

- [How to create tools](https://python.langchain.com/docs/concepts/tools/)
- [Integrating search tool](https://python.langchain.com/docs/integrations/tools/tavily_search/)

### Constraints:

- Use @tool decorator to define the tool
- Use bind_tools() to attach it to the LLM
- Add a ToolNode to handle tool execution
- Use tools_condition as the conditional edge

In [5]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [14]:
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_tavily import TavilySearch
import random


# 1st tool - search
search = TavilySearch(
    max_results=5,
    topic="general",
)

# 2nd tool - Random fact generator
@tool
def random_fact() -> str:
    """Returns a random interesting fact about AI, programming, travel, oceans, or geography."""

    fact = [
        "The term \"artificial intelligence\" was coined at the 1956 Dartmouth workshop.",
        "\"AI effect\" describes how once a task is solved by AI, people stop calling it AI.",
        "Modern LLMs predict the next token rather than \"thinking\" in sentences.",
        "The Turing Test was proposed in 1950 as a measure of machine intelligence.",
        "AI has beaten world champions at chess, Go, and poker.",
        "The first programmer is often considered Ada Lovelace, in the 1840s.",
        "\"Hello, World!\" was popularized by a 1978 C book.",
        "There are 700+ programming languages; most developers use a handful.",
        "The bug-and-debug term came from a real moth found in a 1947 computer.",
        "Python is named after Monty Python, not the snake.",
        "France is the most-visited country in the world.",
        "The shortest commercial flight is ~57 seconds, in Scotland's Orkney Islands.",
        "There's a hotel in Sweden made of ice, rebuilt each winter.",
        "Russia spans 11 time zones.",
        "Japan has more than 5 million vending machines.",
        "Over \"80%\" of the ocean is unexplored.",
        "The Mariana Trench is deeper than Everest is tall.",
        "The ocean produces over half of the world's oxygen.",
        "The blue whale is the largest animal that has ever lived.",
        "At the deepest point, sunlight never reaches."
        ]
    return  random.choice(fact)

In [15]:
tools = [search, random_fact]

# llm with tools
model = model.bind_tools(tools)

In [16]:
# cha function 
def chat(state: State) -> str:
    response = model.invoke(state["messages"])
    return {"messages": [response]}

In [17]:
from langgraph.prebuilt import tools_condition

# workflow
builder = StateGraph(State)
# nodes
builder.add_node("model", chat)
builder.add_node("tools", ToolNode([search, random_fact]))
# edges
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")

In [18]:
checkpoint = InMemorySaver()
store = InMemoryStore()

graph =builder.compile(checkpointer=checkpoint, store=store)

In [19]:
def chat_model(user_input: str, thread_id: str = None) -> str:
    config = {"configurable": {"thread_id": thread_id or str(uuid.uuid4())[:255]}}


    result = graph.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config,
    )
    return result["messages"][-1].content

In [20]:
def run_chat_loop():
    thread_id = str(uuid.uuid4())  # generated ONCE here
    print(f"Session started. Thread ID: {thread_id}")
    print("Welcome to the chat! Type 'exit' to quit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            print("Exiting the chat. Goodbye!")
            break
        response = chat_model(user_input, thread_id)
        print(f"AI: {response}")

In [21]:
run_chat_loop()

Session started. Thread ID: f9c6ad37-52a2-4760-96b6-e1bd1ef1e1a1
Welcome to the chat! Type 'exit' to quit.
AI: The ocean produces over half of the world’s oxygen.
AI: “AI effect” — the phenomenon where once a task is successfully performed by artificial intelligence, people stop referring to it as “AI.”
AI: **Current weather in Thane (Maharashtra, India – as of 3:51 PM on 13 Sept 2026)**  

| Parameter | Value |
|-----------|-------|
| **Temperature** | ~ 26 °C (around 79 °F) |
| **Feels‑like** | 26 °C |
| **Weather** | Isolated showers possible; heavy rain alerts in effect |
| **Precipitation** | 7–12 mm (rainfall) with a high probability of rain (≈ 86‑97 %) |
| **Humidity** | Not listed in the snippet, but typical for the monsoon season (≈ 80 %+) |
| **Wind** | South‑west at ~ 9 km/h, gusts up to 20‑24 km/h |
| **Air Quality Index (AQI)** | 20 – “Good” |
| **Sunrise / Sunset** | 06:25 am / 06:42 pm |
| **Alerts** | Orange alert for heavy rainfall (2:16 PM – 4:00 PM) and a Yellow watc

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01khjrtmmkfjnvd1kgawb2xtfp` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8505, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}